In [ ]:
# 必要ライブラリ
import zipfile
import requests
import pandas as pd
from gensim.models import KeyedVectors
from scipy.stats import spearmanr

# Step 1: WordSim-353 zipファイルをダウンロード
url = "https://www.gabrilovich.com/resources/data/wordsim353/wordsim353.zip"
zip_path = "wordsim353.zip"
csv_path = "combined.csv"

if not os.path.exists(csv_path):
    print("📦 Downloading and extracting WordSim-353 dataset...")
    r = requests.get(url)
    with open(zip_path, "wb") as f:
        f.write(r.content)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()

print("✅ Data ready.")

# Step 2: combined.csv を読み込み
df = pd.read_csv("combined.csv")
df.columns = ["Word1", "Word2", "Human"]  # 念のため明示的にカラム名設定

# Step 3: Word2Vec モデル（すでに読み込んでいる前提）
# model = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

# Step 4: 類似度を計算
similarities = []
skipped_pairs = []

for _, row in df.iterrows():
    w1, w2 = row["Word1"], row["Word2"]
    if w1 in model and w2 in model:
        sim = model.similarity(w1, w2)
        similarities.append(sim)
    else:
        similarities.append(None)
        skipped_pairs.append((w1, w2))

# 有効なデータだけ取り出す
df["ModelSim"] = similarities
df_valid = df.dropna()

# Step 5: スピアマン相関係数の計算
rho, pval = spearmanr(df_valid["Human"], df_valid["ModelSim"])
print(f"📊 Spearman correlation: ρ = {rho:.4f}, p = {pval:.4g}")
print(f"✅ 有効なペア数: {len(df_valid)} / {len(df)}")